In [4]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration

In [5]:
# load datasets
train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")
test_data = pd.read_csv("samsum-test.csv")

In [6]:
train_data.head(5)

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [7]:
# the data is too big for the training so we are not going to train all the data
train_data = train_data.sample(n=4000, random_state=42).reset_index(drop=True)
val_data = val_data.sample(n=500, random_state = 42).reset_index(drop=True)

In [8]:
train_data.shape

(4000, 3)

In [9]:
val_data.shape

(500, 3)

#### Data Preprocessing

In [10]:
import re
def clean_data(text):
  text = re.sub(r"\r\n", " ", text)           # lines
  text = re.sub(r"s+", " ", text)             # spaces
  text = re.sub(r"<.*?>", " ", text)          # html tags
  text = text.strip().lower()                 # spaces and lowercase
  return text

In [11]:
# apply the function
train_data['dialogue'] = train_data['dialogue'].apply(clean_data)
print(train_data['dialogue'])

0       nathan: i'm kinda bored\nnathan: do you have a...
1       daniel: btw have you  tarted watching the  eri...
2       ian: god damn! i'm not gonna make it on time! ...
3       emily: i  aw you at the beach ye terday. what ...
4       matt: good morning :) sophie: hello :) :* matt...
                              ...                        
3995    adam: hey nina how are you? what  up? nina: i ...
3996    jame : i have a propo al for you all nicky: ho...
3997    richard: hey guy, what time will you be here? ...
3998    jim: hi guy  derek: hi andy: hi, man jim: have...
3999    ian: hi gu ! are you free tonight? gu : ye , w...
Name: dialogue, Length: 4000, dtype: object


In [12]:
# clean for the summary col and also for the val_data
train_data['summary'] = train_data['summary'].apply(clean_data)

val_data['dialogue'] = val_data['dialogue'].apply(clean_data)
val_data['summary'] = val_data['summary'].apply(clean_data)

#### Tokenization

In [13]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [15]:
from numpy import True_
# raw data => tokenized inputs for fine-tuning
def tokenize(data):
  inputs = tokenizer(data['dialogue'], padding = 'max_length', max_length = 512, truncation=True)
  targets = tokenizer(data['summary'], padding = 'max_length', max_length = 150, truncation=True)

  inputs['labels'] = targets['input_ids']         # token ids => add to input as labels
  return inputs

In [16]:
train_data = train_data.apply(tokenize, axis=1).tolist()
val_data = val_data.apply(tokenize, axis=1).tolist()

In [18]:
train_data[1]

{'input_ids': [3, 26, 2738, 15, 40, 10, 3, 115, 17, 210, 43, 25, 3, 2046, 1054, 3355, 8, 3, 4074, 780, 58, 3, 7, 1427, 10, 8, 10211, 3, 58, 3, 23, 43, 11, 3, 23, 183, 16, 333, 28, 34, 5, 34, 31, 310, 207, 5, 3, 23, 1243, 3, 23, 8207, 129, 12, 1605, 3, 4074, 24, 453, 975, 3, 23, 5748, 1019, 8, 3, 15, 9, 30, 11, 3, 23, 43, 118, 2701, 397, 3355, 3, 7436, 80, 16, 565, 8, 50, 3, 17, 97, 62, 23004, 5, 3, 26, 2738, 15, 40, 10, 3, 23, 2124, 25, 31, 26, 114, 34, 5, 8, 589, 79, 103, 33, 6139, 6, 132, 31, 150, 194, 12, 817, 149, 186, 151, 79, 31, 162, 4792, 3, 117, 61, 3, 7, 1427, 10, 3, 23, 214, 6, 68, 21, 3, 7159, 3, 864, 30, 34, 103, 15, 3, 29, 31, 17, 143, 25, 114, 135, 90, 3, 6, 3, 23, 317, 24, 31, 8, 36, 3, 17, 294, 81, 34, 55, 3, 26, 2738, 15, 40, 10, 3, 23, 26, 157, 6, 3, 23, 253, 543, 614, 12, 12, 8276, 3, 7159, 715, 3, 233, 16241, 385, 214, 34, 66, 3, 7, 1427, 10, 114, 16, 24, 197, 29, 15, 116, 3, 88, 816, 3, 88, 133, 2870, 3, 18118, 7446, 58, 114, 3, 88, 3, 17, 232, 3, 9, 1253, 55, 3, 